
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Document Parsing and Chunking Strategies for AI Search
## Overview
Before a retrieval agent can answer questions about your documents, those documents must be transformed from raw files into searchable, structured data. This lecture covers the key Databricks AI Functions that power this pipeline: `ai_parse_document` for extracting text from PDFs, `ai_classify` and `ai_extract` for structuring content, and `ai_prep_search` for chunking text for AI Search. We'll also cover how chunks flow into a AI Search index: what happens when you create one, how it stays in sync, and why the text you embed differs from the text you retrieve.

## Learning Objectives
By the end of this lecture, you will be able to:
1. Describe how `ai_parse_document` extracts structured content from PDFs
2. Explain the role of `ai_classify` and `ai_extract` in document processing
3. Use `ai_prep_search` to chunk text for AI Search
4. Compare different chunking strategies and their trade-offs
5. Explain what happens when a AI Search index is created and how it stays in sync
6. Distinguish between `chunk_to_embed` and `chunk_to_retrieve` and why they differ

## A. The Document Processing Pipeline



### A1. From Raw Files to Searchable Data

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 855 245" role="img" style="font-family: sans-serif;">
  <title>Document Processing Pipeline: parallel branches from parsed document</title>
  <defs>
    <marker id="a1c-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Raw PDF -->
  <rect x="10" y="75" width="125" height="95" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="72" y="112" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Raw PDF</text>
  <text x="72" y="132" text-anchor="middle" font-size="15" fill="#618794">UC Volume</text>

  <!-- Arrow to parse -->
  <path d="M135 122 L178 122" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#a1c-arrow)"/>

  <!-- ai_parse_document -->
  <rect x="183" y="58" width="180" height="128" rx="12" fill="#2272B4" fill-opacity="0.12" stroke="#2272B4" stroke-width="1.5"/>
  <text x="273" y="100" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">ai_parse_document</text>
  <text x="273" y="120" text-anchor="middle" font-size="15" fill="#618794">Extract text + layout</text>
  <text x="273" y="140" text-anchor="middle" font-size="15" fill="#618794">-> VARIANT output</text>

  <!-- Fan-out arrows (parallel from parsed doc) -->
  <path d="M363 90 L410 41" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a1c-arrow)"/>
  <path d="M363 122 L410 121" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a1c-arrow)"/>
  <path d="M363 154 L410 201" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a1c-arrow)"/>

  <!-- ai_prep_search (red) -->
  <rect x="418" y="12" width="170" height="58" rx="8" fill="#FF5F46" fill-opacity="0.12" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="503" y="36" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">ai_prep_search</text>
  <text x="503" y="56" text-anchor="middle" font-size="15" fill="#618794">Chunk for VS</text>

  <!-- ai_classify (green) -->
  <rect x="418" y="92" width="170" height="58" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="503" y="116" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">ai_classify</text>
  <text x="503" y="136" text-anchor="middle" font-size="15" fill="#618794">Categorize content</text>

  <!-- ai_extract (green) -->
  <rect x="418" y="172" width="170" height="58" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="503" y="196" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">ai_extract</text>
  <text x="503" y="216" text-anchor="middle" font-size="15" fill="#618794">Pull structured fields</text>

  <!-- Converge arrows (join into enriched chunks) -->
  <path d="M588 41 L648 98" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a1c-arrow)"/>
  <path d="M588 121 L648 121" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a1c-arrow)"/>
  <path d="M588 201 L648 144" fill="none" stroke="#1B3139" stroke-width="1.5" marker-end="url(#a1c-arrow)"/>

  <!-- Enriched Chunk Table (output) -->
  <rect x="653" y="56" width="190" height="132" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="748" y="82" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Enriched Chunks</text>
  <text x="748" y="106" text-anchor="middle" font-size="13" fill="#FF5F46">chunks</text>
  <text x="748" y="124" text-anchor="middle" font-size="13" fill="#00A972">+ category labels</text>
  <text x="748" y="142" text-anchor="middle" font-size="13" fill="#00A972">+ extracted fields</text>
  <text x="748" y="170" text-anchor="middle" font-size="14" fill="#618794">-> AI Search</text>
</svg>
</div>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
How does the document processing pipeline work?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">

<p style="margin-top: 10px;">RAG systems are only as good as the data they retrieve. Before documents can power a RAG agent, they must be processed through a pipeline that extracts, classifies, and chunks the content. Databricks Knowledge Assistant can ingest UC Volumes and tables directly. It runs its own parsing and chunking pipeline under the hood, so you don't need to wire these AI Functions yourself.</p>

<p>Databricks provides a set of AI Functions (<a href="https://docs.databricks.com/aws/en/large-language-models/ai-functions/" style="color: #1B5162; font-weight: 600;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-functions/" style="color: #1B5162; font-weight: 600;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/large-language-models/ai-functions/" style="color: #1B5162; font-weight: 600;">GCP</a>), SQL-callable functions powered by foundation models, that handle each stage. As an example workflow, we might have:</p>

<ul>
<li><code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">ai_parse_document</code> produce a <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">VARIANT</code> output</li>
<li><code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">ai_prep_search</code>, <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">ai_classify</code>, and <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">ai_extract</code> run <strong>in parallel</strong> off that parsed document</li>
<li>Join the classifications and extracted fields onto the chunk table as extra metadata columns for filtering or ranking in AI Search</li>
</ul>

</div>
</details>

## B. Parsing Documents with `ai_parse_document`



### B1. How It Works

`ai_parse_document` ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_parse_document) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_parse_document) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/ai_parse_document)) invokes state-of-the-art generative AI models to extract structured content from PDFs and images. It returns a structured JSON object (VARIANT type) with layout-aware text extraction.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 700 200" role="img" style="font-family: sans-serif;">
  <title>ai_parse_document: from raw PDF to structured output</title>
  <defs>
    <marker id="pd-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Input: Raw PDF -->
  <rect x="15" y="30" width="130" height="130" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="80" y="56" text-anchor="middle" font-size="17" font-weight="600" fill="#0b2026">Raw PDF</text>
  <rect x="30" y="68" width="100" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <rect x="30" y="84" width="80" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <rect x="30" y="100" width="60" height="24" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="60" y="116" text-anchor="middle" font-size="14" fill="#2272B4">table</text>
  <rect x="96" y="100" width="34" height="24" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="113" y="116" text-anchor="middle" font-size="14" fill="#2272B4">img</text>
  <rect x="30" y="130" width="100" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <!-- Arrow -->
  <path d="M145 95 L210 95" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#pd-arrow)"/>
  <!-- Function box -->
  <rect x="215" y="32" width="220" height="128" rx="12" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.5"/>
  <text x="325" y="72" text-anchor="middle" font-size="18" font-weight="700" fill="#0b2026">ai_parse_document</text>
  <text x="325" y="96" text-anchor="middle" font-size="15" fill="#618794">OCR + layout analysis,</text>
  <text x="325" y="116" text-anchor="middle" font-size="15" fill="#618794">figure descriptions,</text>
  <text x="325" y="136" text-anchor="middle" font-size="15" fill="#618794">bounding boxes</text>
  <!-- Arrow -->
  <path d="M435 95 L490 95" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#pd-arrow)"/>
  <!-- Output -->
  <rect x="495" y="30" width="190" height="140" rx="8" fill="#ffffff" stroke="#00A972" stroke-width="1.5"/>
  <text x="590" y="55" text-anchor="middle" font-size="17" font-weight="600" fill="#00A972">Structured JSON</text>
  <text x="514" y="78" font-size="14" fill="#0b2026" font-family="monospace">document:</text>
  <text x="526" y="94" font-size="14" fill="#618794" font-family="monospace">  elements: [...]</text>
  <text x="526" y="110" font-size="14" fill="#618794" font-family="monospace">  pages: [...]</text>
  <text x="514" y="130" font-size="14" fill="#0b2026" font-family="monospace">  error_status: [...]</text>
  <text x="514" y="150" font-size="14" fill="#0b2026" font-family="monospace">metadata: {...}</text>
</svg>
</div>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
What are the four stages of a Knowledge Assistant pipeline?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">

<p style="margin-top: 10px;">Databricks AI Functions transform raw PDFs into searchable data in three stages: <li><strong>Parse (ai_parse_document)</strong>  extracts text and layout as structured JSON</li>
<li><strong>Classify + Extract (ai_classify, ai_extract)</strong>  categorize and pull structured fields</li>
<li><strong>Chunk (ai_prep_search)</strong> splits text into semantic chunks for AI Search.</li>
<li><code>version</code> is an option flag that tells <code>ai_parse_document</code> which schema/output version to use.
<li><strong>v2 ('2.0')</strong> is the recommended version: it gives you the newer output format (HTML tables, improved layout, etc.) and is what v2 <code>ai_extract</code>/<code>ai_classify</code> examples are tuned for.
</li></p>

<pre style="background: #FFFFFF; color: #0b2026; padding: 12px 14px; border: 1px solid #D8D4CA; border-radius: 6px; margin: 10px 0; font-size: 11pt; line-height: 1.5; overflow-x: auto;"><code>SELECT
  path,
  ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
FROM READ_FILES('/Volumes/...', format => 'binaryFile')
</code></pre>
<p style="margin: 6px 0 0 0;">The output includes <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">document:elements</code> (individual content blocks) and <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">document:pages</code> (page-level breakdown), giving you flexible access to the parsed content.</p>
</li>
</ol>

</div>
</details>

## C. Classifying and Extracting with AI Functions
Next, we will look at some examples of how to use `ai_classify()` and `ai_extract()`. Parsing the raw PDF into a structured JSON is straightforward as indicated above. However, we now have to use the table-valude function `variant_explode` ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/variant_explode) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/variant_explode) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/variant_explode)) to un-nest our parsed document.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
How does variant_explode work?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">

<p style="margin-top: 10px;">The <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">variant_explode</code> table-valued function turns a <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">VARIANT</code> array or object into a set of rows with three columns:</p>

<ul>
<li><code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">pos INT</code> &mdash; the position of the element within the array or object</li>
<li><code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">key STRING</code> &mdash; the field name (objects only; <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">NULL</code> for arrays)</li>
<li><code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">value VARIANT</code> &mdash; the field value or array element</li>
</ul>

<p>Behavior depends on the input shape:</p>
<ul>
<li><strong>VARIANT object</strong> &rarr; <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">key</code> and <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">value</code> are the object's keys and values.</li>
<li><strong>VARIANT array</strong> &rarr; <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">key</code> is always <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">NULL</code>; <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">value</code> holds each element.</li>
<li><strong>NULL or non-VARIANT-array/object input</strong> &rarr; <strong>no rows are produced</strong>. To preserve the row with <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">NULL</code> values instead, use variant_explode_outer (<a href="https://docs.databricks.com/aws/en/sql/language-manual/functions/variant_explode_outer" style="color: #1B5162; font-weight: 600;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/variant_explode_outer" style="color: #1B5162; font-weight: 600;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/sql/language-manual/functions/variant_explode_outer" style="color: #1B5162; font-weight: 600;">GCP</a>).</li>
</ul>

<pre style="background: #FFFFFF; color: #0b2026; padding: 12px 14px; border: 1px solid #D8D4CA; border-radius: 6px; margin: 10px 0; font-size: 11pt; line-height: 1.5; overflow-x: auto;"><code>SELECT pos, key, value
FROM variant_explode(parse_json('{"name": "Ada", "age": 36}'));

-- pos | key  | value
-- ----+------+--------
--  0  | name | "Ada"
--  1  | age  | 36
</code></pre>

<p style="margin: 6px 0 0 0;">Use <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">variant_explode</code> as a lateral source (e.g., in a <code style="background: #EEEDE9; padding: 1px 6px; border-radius: 3px; font-size: 11pt;">LATERAL VIEW</code> or comma join) to flatten semi-structured VARIANT columns into queryable rows.</p>

</div>
</details>





### C1. Categorize Content: `ai_classify`

`ai_classify` ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_classify) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_classify) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/ai_classify)) assigns a label to text from a set of categories you define. It's useful for routing documents through different processing paths.
- `ai_parsed_document` produces a `VARIANT` column called `parsed_content`. Passing `parsed_content` lets ai_classify see the full, parsed document instead of just a single text string.
- `["invoice", "receipt", "contract", "report"]` is the set of categories you want the model to choose from.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 720 165" role="img" style="font-family: sans-serif;">
  <title>ai_classify: assigns a category label to text</title>
  <defs>
    <marker id="cl-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Input -->
  <rect x="15" y="18" width="170" height="126" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="100" y="42" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Parsed Text</text>
  <!-- Gray bar above -->
  <rect x="25" y="50" width="150" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <!-- Chip 1 -->
  <rect x="25" y="66" width="150" height="22" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="100" y="81" text-anchor="middle" font-size="11" fill="#2272B4">"Total due: $4,092..."</text>
  <!-- Chip 2 -->
  <rect x="25" y="94" width="150" height="22" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="100" y="109" text-anchor="middle" font-size="11" fill="#2272B4">"Payment terms: Net 30"</text>
  <!-- Gray bar below -->
  <rect x="25" y="122" width="150" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <path d="M185 81 L240 81" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#cl-arrow)"/>
  <!-- Function -->
  <rect x="245" y="37" width="200" height="80" rx="12" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.5"/>
  <text x="345" y="67" text-anchor="middle" font-size="18" font-weight="700" fill="#0b2026">ai_classify</text>
  <text x="345" y="87" text-anchor="middle" font-size="14" fill="#618794">parsed_content,</text>
  <text x="345" y="101" text-anchor="middle" font-size="14" fill="#618794">["invoice",...]</text>
  <path d="M445 77 L505 77" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#cl-arrow)"/>
  <!-- Output -->
  <rect x="510" y="47" width="170" height="56" rx="8" fill="#ffffff" stroke="#00A972" stroke-width="1.5"/>
  <text x="595" y="71" text-anchor="middle" font-size="17" font-weight="600" fill="#00A972">invoice</text>
  <text x="595" y="89" text-anchor="middle" font-size="15" fill="#618794">Single label returned</text>
</svg>
</div>
Sample Code:
<pre style="background: #FFFFFF; color: #0b2026; padding: 12px 14px; border: 1px solid #D8D4CA; border-radius: 6px; margin: 10px 0; font-size: 11pt; line-height: 1.5; overflow-x: auto;"><code>WITH elements AS (
  &amp;lt;read_file_logic&amp;gt; t,
  LATERAL variant_explode(parsed:document:elements) AS element
  &amp;lt;additional_WHERE_logic&amp;gt;
),
classified AS (
  SELECT
    content,
    ai_classify(
      content,
      '["property_description","pricing_information","location_details","amenities","host_information","other"]'
    ) AS cls
  FROM elements
)
SELECT
  content,
  cls:response[0] AS category -- Parse the results from ai_classify()
FROM classified</pre>



### C2. Pull Structured Fields: `ai_extract`

`ai_extract` ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_extract) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_extract) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/ai_extract)) extracts specific fields from unstructured text. You define what to extract, and the AI model finds the values.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 720 180" role="img" style="font-family: sans-serif;">
  <title>ai_extract: pulls structured fields from text</title>
  <defs>
    <marker id="ex-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Input -->
  <rect x="15" y="15" width="170" height="148" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="100" y="40" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Parsed Text</text>
  <!-- Gray bar above -->
  <rect x="25" y="48" width="150" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <!-- Chip 1 -->
  <rect x="25" y="63" width="150" height="22" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="100" y="78" text-anchor="middle" font-size="11" fill="#2272B4">"From: Apex Roofing"</text>
  <!-- Chip 2 -->
  <rect x="25" y="89" width="150" height="22" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="100" y="104" text-anchor="middle" font-size="11" fill="#2272B4">"Date: Jan 15, 2026"</text>
  <!-- Chip 3 -->
  <rect x="25" y="115" width="150" height="22" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="100" y="130" text-anchor="middle" font-size="11" fill="#2272B4">"Total: $4,092.64"</text>
  <!-- Gray bar below -->
  <rect x="25" y="142" width="150" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <path d="M185 89 L240 89" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ex-arrow)"/>
  <!-- Function -->
  <rect x="245" y="44" width="200" height="90" rx="12" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.5"/>
  <text x="345" y="74" text-anchor="middle" font-size="18" font-weight="700" fill="#0b2026">ai_extract</text>
  <text x="345" y="94" text-anchor="middle" font-size="14" fill="#618794">parsed_content,</text>
  <text x="345" y="108" text-anchor="middle" font-size="14" fill="#618794">["invoice_id",...]</text>
  <path d="M445 89 L505 89" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ex-arrow)"/>
  <!-- Output -->
  <rect x="510" y="20" width="190" height="145" rx="8" fill="#ffffff" stroke="#00A972" stroke-width="1.5"/>
  <text x="605" y="43" text-anchor="middle" font-size="16" font-weight="600" fill="#00A972">Structured JSON</text>
  <text x="522" y="66" font-size="13" fill="#0b2026" font-family="monospace">error_message: null</text>
  <text x="522" y="86" font-size="13" fill="#0b2026" font-family="monospace">metadata: {...}</text>
  <text x="522" y="106" font-size="13" fill="#0b2026" font-family="monospace">response:</text>
  <text x="534" y="124" font-size="13" fill="#618794" font-family="monospace">  invoice_id: "1234"</text>
  <text x="534" y="142" font-size="13" fill="#618794" font-family="monospace">  date: "Jan 15, 2026"</text>
  <text x="534" y="160" font-size="13" fill="#618794" font-family="monospace">  total: "$4,092.64"</text>
</svg>
</div>

Sample Code:
<pre style="background: #FFFFFF; color: #0b2026; padding: 12px 14px; border: 1px solid #D8D4CA; border-radius: 6px; margin: 10px 0; font-size: 11pt; line-height: 1.5; overflow-x: auto;"><code>WITH elements AS (
  &amp;lt;read_file_logic&amp;gt; t,
  &amp;lt;variant_explode_logic&amp;gt;
),
extracted AS (
  SELECT
    content,
    ai_extract(
      content,
      '["invoice_id", "date", "total"]'
    ) AS ext
  FROM elements
)
SELECT
  content,
  ext:response.invoice_id   AS invoice_id,
  ext:response.date    AS date,
  ext:response.total   AS total
FROM extracted
LIMIT 15</code></pre>

## D. Chunking with `ai_prep_search`


### D1. What is `ai_prep_search`?

`ai_prep_search` ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_prep_search) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_prep_search) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/ai_prep_search)) is a Databricks AI Function that **chunks text for AI Search**. Instead of manually splitting text by character count or using custom chunking logic, `ai_prep_search` uses AI models to split text into semantically meaningful chunks.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 720 180" role="img" style="font-family: sans-serif;">
  <title>ai_prep_search: chunks text at semantic boundaries for AI Search</title>
  <defs>
    <marker id="ps-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Input -->
  <rect x="15" y="15" width="170" height="148" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="100" y="40" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Parsed Text</text>
  <!-- Gray bar above -->
  <rect x="25" y="48" width="150" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <!-- Chip 1 -->
  <rect x="25" y="63" width="150" height="22" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="100" y="78" text-anchor="middle" font-size="11" fill="#2272B4">"Property is located in..."</text>
  <!-- Chip 2 -->
  <rect x="25" y="89" width="150" height="22" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="100" y="104" text-anchor="middle" font-size="11" fill="#2272B4">"House rules include..."</text>
  <!-- Chip 3 -->
  <rect x="25" y="115" width="150" height="22" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="100" y="130" text-anchor="middle" font-size="11" fill="#2272B4">"Host info: Superhost..."</text>
  <!-- Gray bar below -->
  <rect x="25" y="142" width="150" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
  <path d="M185 89 L240 89" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ps-arrow)"/>
  <!-- Function -->
  <rect x="245" y="44" width="200" height="90" rx="12" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.5"/>
  <text x="345" y="74" text-anchor="middle" font-size="18" font-weight="700" fill="#0b2026">ai_prep_search</text>
  <text x="345" y="94" text-anchor="middle" font-size="14" fill="#618794">AI-powered semantic</text>
  <text x="345" y="108" text-anchor="middle" font-size="14" fill="#618794">boundary detection</text>
  <path d="M445 89 L505 89" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#ps-arrow)"/>
  <!-- Output -->
  <rect x="510" y="20" width="195" height="145" rx="8" fill="#ffffff" stroke="#00A972" stroke-width="1.5"/>
  <text x="607" y="43" text-anchor="middle" font-size="16" font-weight="600" fill="#00A972">Chunk Array</text>
  <rect x="522" y="53" width="172" height="24" rx="6" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1"/>
  <text x="530" y="69" text-anchor="start" font-size="12" fill="#0b2026">Chunk 1: "Property is..."</text>
  <rect x="522" y="83" width="172" height="24" rx="6" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1"/>
  <text x="530" y="99" text-anchor="start" font-size="12" fill="#0b2026">Chunk 2: "House rules..."</text>
  <rect x="522" y="113" width="172" height="24" rx="6" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1"/>
  <text x="530" y="129" text-anchor="start" font-size="12" fill="#0b2026">Chunk 3: "Host info..."</text>
  <!-- Gray bar in output -->
  <rect x="522" y="143" width="172" height="10" rx="2" fill="#618794" fill-opacity="0.3"/>
</svg>
</div>
Sample Code:
<pre style="background: #FFFFFF; color: #0b2026; padding: 12px 14px; border: 1px solid #D8D4CA; border-radius: 6px; margin: 10px 0; font-size: 11pt; line-height: 1.5; overflow-x: auto;"><code>SELECT ai_prep_search(parsed_text) AS chunks
FROM parsed_documents</code></pre>

### D2. Chunking Strategy Considerations

Regardless of the chunking method, keep these principles in mind:

- **Chunk size vs. embedding model limits** - chunks must fit within the embedding model's context window (e.g., 512 tokens for many models). Text beyond the limit is silently truncated.
- **Overlap** - including a small overlap between consecutive chunks (e.g. 10-20%) prevents information from being lost at chunk boundaries.
- **Granularity trade-off** - smaller chunks are more precise for retrieval but may lack surrounding context. Larger chunks capture more context but may dilute the signal.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Lost in the middle</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The "Lost in the Middle" phenomenon means LLMs tend to overlook information buried deep in long contexts. Creating smaller, focused chunks ensures critical details aren't missed during retrieval.
</p>
</div>



### D3. Embed vs. Retrieve: Why They're Different Columns

From `ai_prep_search` we can produce two distinct columns from the same parsed content. Understanding this separation is important before you create a AI Search Index.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 780 310" role="img" style="font-family: sans-serif;">
  <title>chunk_to_embed vs chunk_to_retrieve: different text for different purposes</title>
  <defs>
    <marker id="er-arrow-blue" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
    <marker id="er-arrow-green" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#00A972" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- ai_prep_search output box -->
  <rect x="10" y="80" width="180" height="140" rx="12" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.5"/>
  <text x="100" y="108" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">ai_prep_search</text>
  <text x="100" y="130" text-anchor="middle" font-size="14" fill="#618794">Example output per chunk:</text>
  <text x="100" y="154" text-anchor="middle" font-size="13" font-family="monospace" fill="#0b2026">chunk_to_embed</text>
  <text x="100" y="174" text-anchor="middle" font-size="13" font-family="monospace" fill="#0b2026">chunk_to_retrieve</text>
  <text x="100" y="194" text-anchor="middle" font-size="13" font-family="monospace" fill="#0b2026">chunk_id</text>
  <text x="100" y="210" text-anchor="middle" font-size="13" font-family="monospace" fill="#0b2026">chunk_position</text>

  <!-- Arrow to embed path -->
  <path d="M190 130 L260 60" fill="none" stroke="#2272B4" stroke-width="1.8" marker-end="url(#er-arrow-blue)"/>
  <!-- Arrow to retrieve path -->
  <path d="M190 170 L260 240" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#er-arrow-green)"/>

  <!-- EMBED PATH (top) -->
  <rect x="265" y="10" width="240" height="100" rx="8" fill="#2272B4" fill-opacity="0.10" stroke="#2272B4" stroke-width="1.5"/>
  <text x="385" y="36" text-anchor="middle" font-size="16" font-weight="600" fill="#2272B4">chunk_to_embed</text>
  <text x="385" y="58" text-anchor="middle" font-size="13" fill="#618794">Context-enriched text:</text>
  <text x="385" y="76" text-anchor="middle" font-size="12" font-family="monospace" fill="#0b2026">"Title: House Rules</text>
  <text x="385" y="92" text-anchor="middle" font-size="12" font-family="monospace" fill="#0b2026"> Section: Pets > The fee..."</text>

  <path d="M505 60 L560 60" fill="none" stroke="#2272B4" stroke-width="1.8" marker-end="url(#er-arrow-blue)"/>

  <!-- Embedding model -->
  <rect x="565" y="25" width="195" height="70" rx="8" fill="#ffffff" stroke="#2272B4" stroke-width="1.5"/>
  <text x="662" y="52" text-anchor="middle" font-size="14" font-weight="600" fill="#2272B4">Embedding Model</text>
  <text x="662" y="72" text-anchor="middle" font-size="13" fill="#618794">[0.23, -0.41, 0.08, ...]</text>

  <!-- RETRIEVE PATH (bottom) -->
  <rect x="265" y="190" width="240" height="100" rx="8" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.5"/>
  <text x="385" y="216" text-anchor="middle" font-size="16" font-weight="600" fill="#00A972">chunk_to_retrieve</text>
  <text x="385" y="238" text-anchor="middle" font-size="13" fill="#618794">Raw original text:</text>
  <text x="385" y="256" text-anchor="middle" font-size="12" font-family="monospace" fill="#0b2026">"The fee is $50 per pet</text>
  <text x="385" y="272" text-anchor="middle" font-size="12" font-family="monospace" fill="#0b2026"> per night. Max 2 pets."</text>

  <path d="M505 240 L560 240" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#er-arrow-green)"/>

  <!-- LLM -->
  <rect x="565" y="205" width="195" height="70" rx="8" fill="#ffffff" stroke="#00A972" stroke-width="1.5"/>
  <text x="662" y="228" text-anchor="middle" font-size="14" font-weight="600" fill="#00A972">LLM Context</text>
  <text x="662" y="246" text-anchor="middle" font-size="13" fill="#618794">Generates answer from</text>
  <text x="662" y="262" text-anchor="middle" font-size="13" fill="#618794">clean, original text</text>
</svg>
</div>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
Why not use the same text for both?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">

<p style="margin-top: 10px;">Embedding and retrieval serve different goals:</p>

<ul>
<li><strong>Embedding</strong> needs to answer: <em>"What is this chunk about?"</em> Adding surrounding context (headers, section titles) helps the embedding model place the chunk in the right region of vector space, even if the chunk text alone is ambiguous. A chunk that says "The fee is $50" is meaningless without context &mdash; but prepending "Cleaning Policy > Late Checkout" makes it findable.</li>
<li><strong>Retrieval</strong> needs to answer: <em>"What should the LLM read?"</em> The LLM needs the original text to generate accurate answers. Prepending metadata here would waste context window tokens and could confuse the model with formatting artifacts.</li>
</ul>

<p>This separation is a key retrieval quality technique: <strong>optimize for findability when embedding, optimize for readability when retrieving.</strong></p>

</div>
</details>

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="./Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to explore more about document parsing and chunking strategies? Ask Genie Code. Click on the genie icon <img src="./Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">How can I combine custom parsing strategies with AI Functions?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>


## E. From Chunks to AI Search Index

### E1. What Happens When You Create an Index

Once your chunks are in a Delta table, you create a AI Search index to make them searchable. 

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 855 245" role="img" style="font-family: sans-serif;">
  <title>Index creation lifecycle</title>
  <defs>
    <marker id="e1-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Step 1: Input -->
  <rect x="10" y="58" width="160" height="128" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="90" y="80" text-anchor="middle" font-size="14" font-weight="700" fill="#618794">Step 1</text>
  <text x="90" y="98" text-anchor="middle" font-size="15" font-weight="600" fill="#0b2026">Read Source Table</text>
  <!-- Gray bar above -->
  <rect x="22" y="106" width="136" height="8" rx="2" fill="#618794" fill-opacity="0.3"/>
  <!-- Chip 1 -->
  <rect x="22" y="118" width="136" height="20" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="90" y="132" text-anchor="middle" font-size="10" fill="#2272B4">"Property is located..."</text>
  <!-- Chip 2 -->
  <rect x="22" y="142" width="136" height="20" rx="3" fill="#2272B4" fill-opacity="0.15" stroke="#2272B4" stroke-width="0.8"/>
  <text x="90" y="156" text-anchor="middle" font-size="10" fill="#2272B4">"House rules include..."</text>
  <!-- Gray bar below -->
  <rect x="22" y="166" width="136" height="8" rx="2" fill="#618794" fill-opacity="0.3"/>
  <path d="M170 122 L213 122" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#e1-arrow)"/>
  <!-- Step 2: Compute Embeddings -->
  <rect x="218" y="58" width="180" height="128" rx="12" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.5"/>
  <text x="308" y="82" text-anchor="middle" font-size="14" font-weight="700" fill="#618794">Step 2</text>
  <text x="308" y="102" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Compute</text>
  <text x="308" y="120" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Embeddings</text>
  <text x="308" y="142" text-anchor="middle" font-size="13" fill="#618794">Calls embedding model</text>
  <text x="308" y="158" text-anchor="middle" font-size="13" fill="#618794">on each chunk_text row</text>
  <text x="308" y="176" text-anchor="middle" font-size="13" fill="#00A972">[0.23, -0.41, ...]</text>
  <path d="M398 122 L441 122" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#e1-arrow)"/>
  <!-- Step 3: ANN Index -->
  <rect x="446" y="58" width="180" height="128" rx="12" fill="#00A972" fill-opacity="0.10" stroke="#00A972" stroke-width="1.5"/>
  <text x="536" y="82" text-anchor="middle" font-size="14" font-weight="700" fill="#618794">Step 3</text>
  <text x="536" y="102" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">AI Search (ANN)</text>
  <text x="536" y="120" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Index</text>
  <text x="536" y="142" text-anchor="middle" font-size="13" fill="#618794">Organizes vectors for</text>
  <text x="536" y="158" text-anchor="middle" font-size="13" fill="#618794">fast nearest-neighbor</text>
  <text x="536" y="174" text-anchor="middle" font-size="13" fill="#618794">lookup</text>
  <path d="M626 122 L669 122" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#e1-arrow)"/>
  <!-- Step 4: Output -->
  <rect x="674" y="58" width="170" height="128" rx="8" fill="#ffffff" stroke="#00A972" stroke-width="1.5"/>
  <text x="759" y="82" text-anchor="middle" font-size="14" font-weight="700" fill="#618794">Step 4</text>
  <text x="759" y="102" text-anchor="middle" font-size="16" font-weight="600" fill="#00A972">Store Synced</text>
  <text x="759" y="120" text-anchor="middle" font-size="16" font-weight="600" fill="#00A972">Columns</text>
  <text x="759" y="142" text-anchor="middle" font-size="13" fill="#618794">chunk_id</text>
  <text x="759" y="158" text-anchor="middle" font-size="13" fill="#618794">chunk_text</text>
  <text x="759" y="174" text-anchor="middle" font-size="13" fill="#618794">source_path</text>
</svg>
</div>

**Parameter to pay attention when creating a VS endpoint**

| Parameters | What It Does |
|---|---|
| `endpoint_name` | Which VS endpoint serves the index (shared compute in this course) |
| `source_table_name` | The Delta table containing your chunks |
| `index_name` | Fully-qualified name for your index in Unity Catalog |
| `pipeline_type` | `TRIGGERED` (sync on demand) or `CONTINUOUS` (sync automatically) |
| `primary_key` | Unique row identifier used to track which rows have been processed |
| `embedding_source_column` | Which column to send to the embedding model |
| `embedding_model_endpoint_name` | The model that converts text to vectors (e.g., `databricks-gte-large-en`) |
| `columns_to_sync` | Which columns to store alongside vectors so they can be returned in search results |

This initial sync can take several minutes depending on the number of rows and chunk sizes.

### E2. Keeping the Index in Sync with Change Data Feed

When you write chunks to a Delta table for AI Search, you'll enable **Change Data Feed (CDF)**:

```sql
ALTER TABLE chunks_table SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
```

For Delta Sync indexes on standard endpoints, CDF is required and is what enables incremental sync behavior. With CDF enabled, AI Search processes only rows changed since the last sync, rather than rebuilding the index the way storage-optimized endpoints partially do on each sync.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 780 220" role="img" style="font-family: sans-serif;">
  <title>Change Data Feed: Delta table tracks changes, VS index reads only the diff</title>
  <defs>
    <marker id="cdf-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Delta Table -->
  <rect x="10" y="30" width="190" height="160" rx="8" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <text x="105" y="56" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Delta Table</text>
  <text x="105" y="76" text-anchor="middle" font-size="13" fill="#618794">(chunks)</text>

  <!-- Rows showing changes -->
  <rect x="24" y="90" width="162" height="20" rx="3" fill="#618794" fill-opacity="0.15"/>
  <text x="105" y="104" text-anchor="middle" font-size="12" fill="#618794">existing row (unchanged)</text>
  <rect x="24" y="114" width="162" height="20" rx="3" fill="#00A972" fill-opacity="0.2" stroke="#00A972" stroke-width="0.8"/>
  <text x="105" y="128" text-anchor="middle" font-size="12" fill="#00A972">+ new row (INSERT)</text>
  <rect x="24" y="138" width="162" height="20" rx="3" fill="#2272B4" fill-opacity="0.2" stroke="#2272B4" stroke-width="0.8"/>
  <text x="105" y="152" text-anchor="middle" font-size="12" fill="#2272B4">~ updated row (UPDATE)</text>
  <rect x="24" y="162" width="162" height="20" rx="3" fill="#FF5F46" fill-opacity="0.2" stroke="#FF5F46" stroke-width="0.8"/>
  <text x="105" y="176" text-anchor="middle" font-size="12" fill="#FF5F46">- deleted row (DELETE)</text>

  <!-- Arrow: CDF change log -->
  <path d="M200 110 L280 110" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#cdf-arrow)"/>
  <text x="240" y="100" text-anchor="middle" font-size="12" font-weight="600" fill="#618794">CDF log</text>

  <!-- CDF Change Log -->
  <rect x="285" y="50" width="200" height="120" rx="8" fill="#FF5F46" fill-opacity="0.06" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="385" y="76" text-anchor="middle" font-size="15" font-weight="600" fill="#0b2026">Change Data Feed</text>
  <text x="385" y="98" text-anchor="middle" font-size="13" fill="#618794">Records only the diff:</text>
  <text x="385" y="118" text-anchor="middle" font-size="12" font-family="monospace" fill="#0b2026">_change_type</text>
  <text x="385" y="136" text-anchor="middle" font-size="12" font-family="monospace" fill="#0b2026">_commit_version</text>
  <text x="385" y="154" text-anchor="middle" font-size="12" font-family="monospace" fill="#0b2026">_commit_timestamp</text>

  <!-- Arrow: sync -->
  <path d="M485 110 L555 110" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#cdf-arrow)"/>
  <text x="520" y="100" text-anchor="middle" font-size="12" font-weight="600" fill="#618794">sync</text>

  <!-- VS Index -->
  <rect x="560" y="40" width="200" height="140" rx="8" fill="#00A972" fill-opacity="0.08" stroke="#00A972" stroke-width="1.5"/>
  <text x="660" y="68" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">AI Search</text>
  <text x="660" y="86" text-anchor="middle" font-size="16" font-weight="600" fill="#0b2026">Index</text>
  <text x="660" y="110" text-anchor="middle" font-size="13" fill="#618794">Processes only changed</text>
  <text x="660" y="126" text-anchor="middle" font-size="13" fill="#618794">rows, not the full</text>
  <text x="660" y="142" text-anchor="middle" font-size="13" fill="#618794">table on every sync</text>
  <text x="660" y="168" text-anchor="middle" font-size="12" fill="#00A972" font-weight="600">Fast incremental updates</text>
</svg>
</div>

**Sync modes** control *when* those changes flow to the index:

| Mode | Behavior | Trade-off |
|---|---|---|
| **`TRIGGERED`** | Syncs only when you explicitly call `index.sync()` or through the UI | You control when updates happen, which is good for batch workflows where you update chunks periodically |
| **`CONTINUOUS`** | Syncs automatically as the source table changes | Near-real-time freshness, but uses more compute |

## F. Conclusion

In this lecture, we covered the full pipeline from raw documents to searchable AI Search index:

1. **`ai_parse_document`** extracts structured text from raw PDFs with layout awareness
2. **`ai_classify`** and **`ai_extract`** categorize and structure parsed content
3. **`ai_prep_search`** chunks text using AI-powered semantic boundaries, producing separate `chunk_to_embed` (context-enriched for findability) and `chunk_to_retrieve` (raw text for LLM readability) columns
4. **Index creation** reads your source table, computes embeddings via a managed model, and builds an ANN structure for fast similarity search
5. **Change Data Feed** enables efficient incremental sync so the index stays up to date without reprocessing every row

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
